# 01 · Raw Data Exploration

Author: Chris Kim — 2026-05, ChainSense · UCSD DS3

Two things only:

1. **What the raw BigQuery dump looks like** — the fields and how they're organized.
2. **Bronze → Silver** — how the sharded BigQuery export is consolidated and partitioned with DuckDB.


## 1. The raw BigQuery dump

Google's bigquery-public-data.crypto_ethereum dataset stores the chain **already flattened** into separate tables:

- **blocks** — one row per block (header fields: number, timestamp, miner, gas_used, base_fee_per_gas, …).
- **transactions** — one row per transaction, **with the receipt joined in**: alongside from_address / to_address / value you also get receipt_gas_used, receipt_status, receipt_contract_address, receipt_effective_gas_price, …

A raw node (here via Alchemy) does **not** pre-join any of this. With web3.py:

| Want | BigQuery | Node / Alchemy RPC |
|------|----------|--------------------|
| block + its txs | SELECT * FROM blocks / transactions | w3.eth.get_block("latest", full_transactions=True) |
| receipts | already on the transactions row | a **second** call: w3.eth.get_transaction_receipt(tx_hash) per tx, or Alchemy's batched eth_getBlockReceipts |

So get_block returns the block and its transactions but **no** gas_used / status — those live on the receipt, which BigQuery folds into the transactions table for you. Field names differ too (snake_case in BQ, camelCase over RPC), as the side-by-side below shows.

> BigQuery access needs gcloud auth application-default login first.


In [1]:
# import relevant libraries and set up BigQuery client and RPC function
from google.cloud import bigquery
client = bigquery.Client(project="chainsense-497110")

In [2]:
# Block from BigQuery
sample = client.query("""
    SELECT * FROM `bigquery-public-data.crypto_ethereum.blocks`
    ORDER BY number DESC
    LIMIT 1
""").to_dataframe()
bq_df = (
    sample.T # Take 1 row sample (column major) and convert into row major
    .reset_index() # after transpose, the original column names are now the index of the dataframe
    .rename(columns={"index": "field", 0: "value"}) # 0 because after transpose the single row had label 0
    .assign(type=lambda df: df["value"].apply(lambda v: type(v).__name__))
    [["field", "type", "value"]] # reorder columns to be more intuitive
)

/Users/mr.po/Documents/GitHub/ucsd-ChainSense/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [3]:
import os, dotenv, pandas as pd
from web3 import Web3
dotenv.load_dotenv()
w3 = Web3(Web3.HTTPProvider(f"https://eth-mainnet.g.alchemy.com/v2/{os.getenv('ALCHEMY_KEY')}"))
block = w3.eth.get_block("latest", full_transactions=True)
rpc_df = pd.DataFrame([(k,type(v).__name__,v) for k,v in block.items()], columns=["field", "type", "value"])

In [4]:
# Big Query Blocks vs RPC (eth_getBlockByNumber) Blocks
pd.concat([bq_df, rpc_df], axis=1, keys=["BigQuery------------------------------------------------------------------------------------", "Alchemy------------------------------------------------------------------------------------"])

BigQuery------------------------------------------------------------------------------------  \
                                                                                          field   
0                                           timestamp                                             
1                                              number                                             
2                                                hash                                             
3                                         parent_hash                                             
4                                               nonce                                             
5                                         sha3_uncles                                             
6                                          logs_bloom                                             
7                                   transactions_root                                             
8                                          state_root                                             
9                                       receipts_root                                             
10                                              miner                                             
11                                         difficulty                                             
12                                   total_difficulty                                             
13                                               size                                             
14                                         extra_data                                             
15                                          gas_limit                                             
16                                           gas_used                                             
17                                  transaction_count                                             
18                                   base_fee_per_gas                                             
19                                   withdrawals_root                                             
20                                        withdrawals                                             
21                                      blob_gas_used                                             
22                                    excess_blob_gas                                             
23                                                NaN                                             
24                                                NaN                                             
25                                                NaN                                             

                                                                  \
         type                                              value   
0   Timestamp                          2026-05-30 23:59:59+00:00   
1         int                                           25211628   
2         str  0x0421f2034c4940215289beaae5cf1b3a7b38e5c0f244...   
3         str  0xe7100d3e395b352a62dcc1f5f13e520a476bebc47464...   
4         str                                 0x0000000000000000   
5         str  0x1dcc4de8dec75d7aab85b567b6ccd41ad312451b948a...   
6         str  0x18ebcf6dff73fdf0ffdde77fe7dbfd9dff98cf3b7fa7...   
7         str  0xc64ce746d4ca1d8e2e326a37e5f6caf90ff6e753c336...   
8         str  0xa275dc31c3adcefe30960b8b7f940ded2970121cca99...   
9         str  0xa1e16b20097d258062f6c5ddcd0044621f175650bb49...   
10        str         0xfb74767c1ce1aada0a0e114441173b57f8c1571b   
11    Decimal                                               0E-9   
12   NoneType                                               None   
13        int                                             261642   
14        str  0x457572656b612028657572656b616275696c6465722e...   
15        int                                           60000000   
16    

### Transactions: receipt fields appended

The BigQuery transactions row carries the receipt_* columns inline; the RPC transaction object does not — note the absence of gas_used / status on the Alchemy side.


In [14]:
day = sample["timestamp"].iloc[0].date()  # day of the latest block we just pulled (guaranteed to have txs)
bq_tx = client.query(f"""
    SELECT * FROM `bigquery-public-data.crypto_ethereum.transactions`
    WHERE DATE(block_timestamp) = '{day}'
    LIMIT 1
""").to_dataframe()
bq_tx = bq_tx.T.reset_index().rename(columns={"index": "field", 0: "value"}).assign(type=lambda df: df["value"].apply(lambda v: type(v).__name__))[["field", "type", "value"]]
rpc_tx = block["transactions"][0]
rpc_tx = pd.DataFrame([(k,type(v).__name__,v) for k,v in rpc_tx.items()], columns=["field", "type", "value"])

/Users/mr.po/Documents/GitHub/ucsd-ChainSense/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [15]:
pd.concat([bq_tx, rpc_tx], axis=1)

,field,type,value,field,type,value
0,hash,str,0xbf539b1605df7ed54ad41e5ff451f55f239067aea93c...,type,int,2
1,nonce,int,243,chainId,int,1
2,transaction_index,int,150,nonce,int,6337497
3,from_address,str,0x1d3b9ffa54d5546a5ce3d828ec16f8b908c1bc09,gas,int,121597
4,to_address,str,0x8dfbdac28503957479857257bddf4178a2636d84,maxFeePerGas,int,124491316
5,value,Decimal,426240508157117800.000000000,maxPriorityFeePerGas,int,4
6,gas,int,21000,to,str,0x1f2F10D1C40777AE1Da742455c65828FF36Df387
7,gas_price,int,212460904,value,int,1323739127
8,input,str,0x,accessList,list,[{'address': '0x43A96962254855F16b925556f9e97B...
9,receipt_cumulative_gas_used,int,11796593,input,HexBytes,"b""\x04\x1a\xba\xa4\x8e\xf5/\xe0\xcf:&\x82\xc1$..."


## 2. Bronze → Silver with DuckDB

A BigQuery extract of this size doesn't come out as one file — it's **sharded** into many parts (transactions-000000000000.parquet, -000000000001.parquet, …). That's the **bronze** layer: raw, sharded, unpartitioned.

DuckDB reads an entire shard glob in a single query and rewrites it into the **silver** layer:

- **contracts** → one deduped, sorted parquet file.
- **transactions** and **blocks** → repartitioned by year=YYYY/month=MM (Hive layout), so downstream feature queries scan only the months they need.


In [ ]:
import duckdb
from pathlib import Path

BRONZE_GLOB = "../data/bronze/contracts/contracts*.parquet"
SILVER_DIR  = "../data/silver/contracts"
SILVER_FILE = f"{SILVER_DIR}/contracts.parquet"

Path(SILVER_DIR).mkdir(parents=True, exist_ok=True)

con = duckdb.connect()

query = f"""
COPY (
    SELECT DISTINCT
        lower(address) AS address   -- normalize case (defensive; BQ exports are lowercase)
    FROM read_parquet('{BRONZE_GLOB}')
    WHERE address IS NOT NULL
    ORDER BY address                -- sorted for better compression + faster joins
)
TO '{SILVER_FILE}'
(
    FORMAT PARQUET,
    COMPRESSION ZSTD,
    ROW_GROUP_SIZE 100000
);
"""

con.execute(query)

# Sanity check
n_contracts = con.execute(f"SELECT COUNT(*) FROM '{SILVER_FILE}'").fetchone()[0]
file_size_mb = Path(SILVER_FILE).stat().st_size / 1024 / 1024
print(f"Wrote {n_contracts:,} unique contract addresses to {SILVER_FILE} ({file_size_mb:.1f} MB)")

In [ ]:
import duckdb
from pathlib import Path

SRC = "../data/bronze/transactions/transactions-*.parquet"
DST = "../data/silver/transactions"

Path(DST).mkdir(parents=True, exist_ok=True)

con = duckdb.connect()

con.execute(f"""
    COPY (
        SELECT
            tx_hash,
            block_number,
            block_timestamp,
            from_address,
            to_address,
            value_wei,
            gas_limit,
            gas_used,
            gas_price_wei,
            effective_gas_price_wei,
            status,
            method_selector,
            CAST(strftime(block_timestamp, '%Y') AS INTEGER) AS year,
            CAST(strftime(block_timestamp, '%m') AS INTEGER) AS month
        FROM read_parquet('{SRC}')
    )
    TO '{DST}'
    (FORMAT PARQUET,
     PARTITION_BY (year, month),
     COMPRESSION ZSTD,
     OVERWRITE_OR_IGNORE);
""")

print("Done. Partitions:")
for p in sorted(Path(DST).glob("year=*/month=*")):
    size_gb = sum(f.stat().st_size for f in p.rglob("*.parquet")) / 1e9
    print(f"  {p.name}: {size_gb:.2f} GB")

print("\nRow counts:")
result = con.execute(f"""
    SELECT year, month, COUNT(*) AS tx_count
    FROM read_parquet('{DST}/**/*.parquet', hive_partitioning=true)
    GROUP BY year, month
    ORDER BY year, month
""").fetchall()

for year, month, count in result:
    print(f"  {year}-{month:02d}: {count:,} txs")

Done. Partitions:
  month=10: 4.08 GB
  month=11: 3.84 GB
  month=12: 4.29 GB
  month=5: 1.20 GB
  month=6: 3.65 GB
  month=7: 4.04 GB
  month=8: 4.48 GB
  month=9: 4.18 GB
  month=1: 5.98 GB
  month=2: 5.34 GB
  month=3: 5.91 GB
  month=4: 6.32 GB
  month=5: 3.70 GB

Row counts:
  2025-05: 13,855,812 txs
  2025-06: 41,950,663 txs
  2025-07: 46,669,810 txs
  2025-08: 51,770,534 txs
  2025-09: 48,896,815 txs
  2025-10: 48,010,794 txs
  2025-11: 45,672,422 txs
  2025-12: 51,471,180 txs
  2026-01: 70,009,743 txs
  2026-02: 61,889,998 txs
  2026-03: 68,542,985 txs
  2026-04: 72,827,602 txs
  2026-05: 44,194,750 txs


In [ ]:
import duckdb
from pathlib import Path

BRONZE_GLOB = "../data/bronze/blocks/blocks*.parquet"  # matches single file or shards
SILVER_DIR  = "../data/silver/blocks"

Path(SILVER_DIR).mkdir(parents=True, exist_ok=True)

con = duckdb.connect()

query = f"""
COPY (
    SELECT
        block_number,
        base_fee_per_gas_wei,
        -- block_timestamp,  -- uncomment if you want to keep it in the silver file
        CAST(strftime(block_timestamp, '%Y') AS INTEGER) AS year,
        CAST(strftime(block_timestamp, '%m') AS INTEGER) AS month
    FROM read_parquet('{BRONZE_GLOB}')
)
TO '{SILVER_DIR}'
(
    FORMAT PARQUET,
    PARTITION_BY (year, month),
    OVERWRITE_OR_IGNORE,
    COMPRESSION ZSTD,
    ROW_GROUP_SIZE 100000
);
"""

con.execute(query)
print("Done. Output written to", SILVER_DIR)

# Sanity check
result = con.execute(f"""
    SELECT year, month, COUNT(*) AS block_count, MIN(block_number) AS min_blk, MAX(block_number) AS max_blk
    FROM read_parquet('{SILVER_DIR}/**/*.parquet', hive_partitioning=true)
    GROUP BY year, month
    ORDER BY year, month
""").fetchall()

for row in result:
    print(f"  {row[0]}-{row[1]:02d}: {row[2]:,} blocks  (block {row[3]:,}{row[4]:,})")

Done. Output written to ../data/silver/blocks
  2025-05: 71,445 blocks  (block 22,534,69822,606,142)
  2025-06: 214,531 blocks  (block 22,606,14322,820,673)
  2025-07: 221,840 blocks  (block 22,820,67423,042,513)
  2025-08: 222,052 blocks  (block 23,042,51423,264,565)
  2025-09: 214,678 blocks  (block 23,264,56623,479,243)
  2025-10: 221,523 blocks  (block 23,479,24423,700,766)
  2025-11: 214,154 blocks  (block 23,700,76723,914,920)
  2025-12: 221,132 blocks  (block 23,914,92124,136,052)
  2026-01: 222,240 blocks  (block 24,136,05324,358,292)
  2026-02: 200,575 blocks  (block 24,358,29324,558,867)
  2026-03: 222,159 blocks  (block 24,558,86824,781,026)
  2026-04: 215,341 blocks  (block 24,781,02724,996,367)
  2026-05: 150,704 blocks  (block 24,996,36825,147,071)


In [17]:
# Resulting silver layout (year=YYYY/month=MM partitions + the single contracts file)
!find ../data/silver -maxdepth 3 -type d | sort
!echo && ls -1 ../data/silver/contracts/


../data/silver
../data/silver/blocks
../data/silver/blocks/year=2025
../data/silver/blocks/year=2025/month=10
../data/silver/blocks/year=2025/month=11
../data/silver/blocks/year=2025/month=12
../data/silver/blocks/year=2025/month=5
../data/silver/blocks/year=2025/month=6
../data/silver/blocks/year=2025/month=7
../data/silver/blocks/year=2025/month=8
../data/silver/blocks/year=2025/month=9
../data/silver/blocks/year=2026
../data/silver/blocks/year=2026/month=1
../data/silver/blocks/year=2026/month=2
../data/silver/blocks/year=2026/month=3
../data/silver/blocks/year=2026/month=4
../data/silver/blocks/year=2026/month=5
../data/silver/contracts
../data/silver/token_transfers
../data/silver/token_transfers/year=2025
../data/silver/token_transfers/year=2025/month=10
../data/silver/token_transfers/year=2025/month=11
../data/silver/token_transfers/year=2025/month=12
../data/silver/token_transfers/year=2025/month=5
../data/silver/token_transfers/year=2025/month=6
../data/silver/token_transfers/